In [1]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=2"

from jVMC_exp.sharding_config import MESH, DEVICE_SHARDING
import jax
import jax.numpy as jnp

print("Devices =", jax.devices())
print("Device count =", jax.device_count())
print("Mesh shape =", MESH.shape["devices"])

Running in single-node mode (JVMC_USE_DISTRIBUTED not set)
Devices = [CpuDevice(id=0), CpuDevice(id=1)]
Device count = 2
Mesh shape = 2


In [2]:
import jax
# jax.config.update("jax_log_compiles", True)
import jax.numpy as jnp

from jVMC_exp.vqs import NQS
from jVMC_exp.sharding_config import sharded
import jVMC_exp


In [3]:
class TestNQS(NQS):
    @sharded(automatic_sharding=True, yield_iter=True)
    def _gradients_iter_sh(self, s, *, parameters, batch_size):
        return self.flat_gradient_function(self.apply_fun, parameters, s)

    def iterative_gradients(self, s):
        return self._gradients_iter_sh(s, parameters=self.grad_parameters, batch_size=self.batchSize)

In [4]:
L = 10
n_samples = 2**8
n_chains = 2**6
batch_size = 2**4

model = jVMC_exp.nets.RBM(L)
psi = NQS(model, L, batch_size, seed=123)
sampler = jVMC_exp.sampler.MCSampler(
    psi, 
    jVMC_exp.propose.SpinFlip(),
    123, 
    n_chains,
    n_samples    
)

samples = sampler.samples

In [10]:
grad = jVMC_exp.stats.SampledObs(psi.gradients(samples), sampler.weights)
grad_lazy = jVMC_exp.stats.LazySampledObs(psi.lazy_gradients(samples), sampler.weights)

In [19]:
grad_covar_obs = grad.get_covar_obs()
grad_covar_mean = grad_lazy.get_covar()
grad_covar_simple = grad_lazy._get_covar_obs()

In [17]:
jnp.allclose(grad_covar_mean, grad_covar_obs.mean)

Array(True, dtype=bool)

In [30]:
grad_covar_simple.var[0][0], grad_covar_obs.var[0][0]

(Array(0.00031848, dtype=float64), Array(0.1358153, dtype=float64))